In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

import torch
import torchvision.transforms as transforms

from PIL import Image
import matplotlib.pyplot as plt

import requests
from io import BytesIO

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Capstone DDS")

sys.path.insert(0, str(PROJECT_ROOT))

MODEL_DIR = PROJECT_ROOT / "models"

print(PROJECT_ROOT)

In [ ]:
from src.model import HybridDeepfakeDetector
from src.transforms import FrequencyTransform

print("Imports Successful!")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
IMAGE_SIZE = 224

rgb_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

frequency_transform = transforms.Compose([
    FrequencyTransform(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor()
])

In [ ]:
model = HybridDeepfakeDetector(
    pretrained=False
).to(device)

checkpoint = torch.load(
    MODEL_DIR/"best_model.pth",
    map_location=device
)

model.load_state_dict(checkpoint)

model.eval()

print("✅ Model Loaded")

In [ ]:
def load_image(source):

    # URL
    if isinstance(source, str) and source.startswith("http"):

        response = requests.get(source, timeout=20)
        response.raise_for_status()

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        return image

    # Local Path
    elif isinstance(source, str):

        image = Image.open(source).convert("RGB")

        return image

    # PIL Image
    elif isinstance(source, Image.Image):

        return source

    else:

        raise ValueError("Unsupported Input")

In [ ]:
def predict(source):

    image = load_image(source)

    rgb = rgb_transform(image).unsqueeze(0).to(device)

    freq = frequency_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(rgb, freq)

        probs = torch.softmax(outputs, dim=1)

        pred = outputs.argmax(1).item()

    real_prob = probs[0][0].item()
    fake_prob = probs[0][1].item()

    confidence = max(real_prob, fake_prob)

    label = "Real" if pred == 0 else "Fake"

    return image, label, confidence, real_prob, fake_prob

In [ ]:
def show_prediction(source):

    image, label, confidence, real_prob, fake_prob = predict(source)

    plt.figure(figsize=(6,6))

    plt.imshow(image)

    plt.axis("off")

    plt.title(
        f"Prediction : {label}\n"
        f"Confidence : {confidence*100:.2f}%"
    )

    plt.show()

    print("="*50)

    print("Prediction :", label)

    print(f"Confidence : {confidence*100:.2f}%")

    print(f"Real Probability : {real_prob*100:.2f}%")

    print(f"Fake Probability : {fake_prob*100:.2f}%")

    print("="*50)

In [ ]:
IMAGE_PATH = "/content/drive/MyDrive/images-test/test1.jpg"

show_prediction(IMAGE_PATH)

In [ ]:
IMAGE_URL = "url"

show_prediction(IMAGE_URL)